In [7]:
# this script merges the raw data until 2025 Q2 with the files for 2025 Q3


import pandas as pd
import os

input1 = "../../50 KM Group/Royalties/Statements/Karen/Earth/Combined statements/old/Earth_2022Q4_2025Q3_1_raw_combined.csv"
input2 = "../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/Earth_2025 Q4_raw_combined.csv"
outputfilename = "../../50 KM Group/Royalties/Statements/Karen/_output/Earth_2022Q4_2025Q4_1_raw_combined.csv"

df_1 = pd.read_csv(input1,low_memory=False)

# three lines below only needed for importing modified 2024Q4 file
#df_1 = df_1.rename(columns={'Statement': 'Statement Quarter'})
#df_1.fillna({'Sales Month': 'XX_UNKNOWN'}, inplace=True)
#df_1["Sales Month"] = df_1["Sales Month"].replace("2453 1月", "XX_UNKNOWN")

print(f"DataFrame: Rows: {df_1.shape[0]}, Columns: {df_1.shape[1]}")
print(f"Total fee: {df_1['Royalties (USD)'].sum()}. Total units: {df_1['Units'].sum()}")

df_2 = pd.read_csv(input2,low_memory=False)
print(f"DataFrame: Rows: {df_2.shape[0]}, Columns: {df_2.shape[1]}")
print(f"Total fee: {df_2['Royalties (USD)'].sum()}. Total units: {df_2['Units'].sum()}")

def align_columns(df1, df2):
    missing_in_df1 = df2.columns.difference(df1.columns)
    missing_in_df2 = df1.columns.difference(df2.columns)    
    
    for col in missing_in_df1:
        df1[col] = pd.NA
    for col in missing_in_df2:
        df2[col] = pd.NA
        
    return df1, df2

def convert_to_majority_type(df):
    df_copy = df.copy()
    
    for col in df_copy.columns:
        # Get the majority type
        type_counts = df_copy[col].map(type).value_counts()
        majority_type = type_counts.idxmax()
        
        # Define the conversion function
        if majority_type == str:
            df_copy[col] = df_copy[col].astype(str)
        elif majority_type == float:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
        elif majority_type == int:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').astype('Int64')
        elif majority_type == bool:
            df_copy[col] = df_copy[col].astype(bool)
        else:
            # Fallback: use string conversion
            df_copy[col] = df_copy[col].astype(str)
    
    return df_copy


print('columns in df1:')
for column in df_1.columns:        
        print(column)

print('columns in df2:')
for column in df_2.columns:        
        print(column)


DataFrame: Rows: 1420784, Columns: 61
Total fee: 570068.9655407665. Total units: 322864915.0
DataFrame: Rows: 135919, Columns: 48
Total fee: 40106.049627258246. Total units: 31877005
columns in df1:
Album
Apple Identifier
Artist
Composer Name
Content
Copyright holder unique code
Country
Currency
Device
FX Rate
Gross Amount
ISRC
Incentivized Ads - Monthly Share of Revenue
Incentivized Ads - Total Plays
Internal costs
Issuance
Noise Content
Non-incentivized Ads - Monthly Share of Revenue
Non-incentivized Ads - Total Downloads
Non-incentivized Ads - Total Plays
Paid/not paid
Period
Period end
Period start
Platform
Play Type
Price
Product
Product Type Identifier
Revenue
Royalties
Royalties (CNY)
Royalties (USD)
Royalty Rate
Sales Month
Sales Quarter
Sales Type
Sales or Return
Sales price
Settlement type
Share
Share Lyricist
Share MABB (CNY)
Share MABB (USD)
Share Master Owner
Share Performer
Share composer
Song
Song ID
Song Length
Statement Quarter
Streaming Subscription Category
Streaming

In [8]:
# print(df_1["Statement Quarter"].map(type).value_counts())

def clean_up_dtype (df,df_name):
    print(f"\nChecking what data types there are in {df_name}")
    for col in df.columns:
        print(f"Column: {col}")
        print(df[col].map(type).value_counts())
        print()
    dfc = convert_to_majority_type(df)
    print(f"\nConverting data types in {df_name}")
    for col in dfc.columns:
        print(f"Column: {col}")
        print(dfc[col].map(type).value_counts())
        print()
    return dfc

df_1c = clean_up_dtype(df_1,input1)
df_2c = clean_up_dtype(df_2,input2)

df_2c["Sales Month"] = df_2c["Sales Month"].astype("string")



Checking what data types there are in ../../50 KM Group/Royalties/Statements/Karen/Earth/Combined statements/old/Earth_2022Q4_2025Q3_1_raw_combined.csv
Column: Album
Album
<class 'str'>      947409
<class 'float'>    473375
Name: count, dtype: int64

Column: Apple Identifier
Apple Identifier
<class 'float'>    1420784
Name: count, dtype: int64

Column: Artist
Artist
<class 'str'>      1420783
<class 'float'>          1
Name: count, dtype: int64

Column: Composer Name
Composer Name
<class 'float'>    1420784
Name: count, dtype: int64

Column: Content
Content
<class 'str'>    1420784
Name: count, dtype: int64

Column: Copyright holder unique code
Copyright holder unique code
<class 'float'>    1420784
Name: count, dtype: int64

Column: Country
Country
<class 'str'>      1419994
<class 'float'>        790
Name: count, dtype: int64

Column: Currency
Currency
<class 'str'>      1354332
<class 'float'>      66452
Name: count, dtype: int64

Column: Device
Device
<class 'float'>    1420784
Na

In [9]:
# Find common columns
common_cols = df_1c.columns.intersection(df_2c.columns)

# Dictionary to hold comparison results
type_comparison = {}

# Loop through each common column
for col in common_cols:
    # Get the set of types present in each column (ignoring NaN)
    types_df1 = set(df_1c[col].dropna().map(type))
    types_df2 = set(df_2c[col].dropna().map(type))
    
    # Store in the results dictionary
    type_comparison[col] = {
        "df1_types": types_df1,
        "df2_types": types_df2,
        "types_match": types_df1 == types_df2
    }

# Convert to a DataFrame for nicer display
type_comparison_df = pd.DataFrame(type_comparison).T

print(type_comparison_df)


                                                         df1_types  \
Album                                              {<class 'str'>}   
Apple Identifier                                 {<class 'float'>}   
Artist                                             {<class 'str'>}   
Composer Name                                                   {}   
Content                                            {<class 'str'>}   
Country                                            {<class 'str'>}   
Currency                                           {<class 'str'>}   
FX Rate                                          {<class 'float'>}   
Gross Amount                                     {<class 'float'>}   
ISRC                                               {<class 'str'>}   
Incentivized Ads - Monthly Share of Revenue      {<class 'float'>}   
Incentivized Ads - Total Plays                   {<class 'float'>}   
Issuance                                         {<class 'float'>}   
Noise Content       

In [10]:
df_1c, df_2c = align_columns(df_1c, df_2c)
df_final = pd.concat([df_1c, df_2c], ignore_index=True)
print(f"Total fee: {df_final['Royalties (USD)'].sum()}. Total units: {df_final['Units'].sum()}")

df_final = df_final.sort_index(axis=1)

print('columns in df_final:')
for column in df_final.columns:        
        print(column)

print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")
df_final = df_final.drop(columns=['实际分成收入(TWD)','总计','Payable CNY'],errors = 'ignore')
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")



/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43759/2446965119.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_1c, df_2c], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43759/2446965119.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_1c, df_2c], ignore_index=True)


Total fee: 610175.0151680243. Total units: 354741920.0
columns in df_final:
Album
Apple Identifier
Artist
Composer Name
Content
Copyright holder unique code
Country
Currency
Device
FX Rate
Gross Amount
ISRC
Incentivized Ads - Monthly Share of Revenue
Incentivized Ads - Total Plays
Internal costs
Issuance
Noise Content
Non-incentivized Ads - Monthly Share of Revenue
Non-incentivized Ads - Total Downloads
Non-incentivized Ads - Total Plays
Paid/not paid
Period
Period end
Period start
Platform
Play Type
Price
Product
Product Type Identifier
Revenue
Royalties
Royalties (CNY)
Royalties (USD)
Royalty Rate
Sales Month
Sales Quarter
Sales Type
Sales or Return
Sales price
Settlement type
Share
Share Lyricist
Share MABB (CNY)
Share MABB (USD)
Share Master Owner
Share Performer
Share composer
Song
Song ID
Song Length
Statement Quarter
Streaming Subscription Category
Streaming Subscription Type
Streaming category
Streaming type
Total Downloads
UPC
Unit Price
Units
Withholding Tax
备注
税前金额 TWD
预扣税TW

In [11]:
for col in df_final.columns:
    print(f"Column: {col}")
    print(df_final[col].map(type).value_counts())
    print()

Column: Album
Album
<class 'str'>    1556703
Name: count, dtype: int64

Column: Apple Identifier
Apple Identifier
<class 'float'>    1556703
Name: count, dtype: int64

Column: Artist
Artist
<class 'str'>    1556703
Name: count, dtype: int64

Column: Composer Name
Composer Name
<class 'float'>    1556703
Name: count, dtype: int64

Column: Content
Content
<class 'str'>    1556703
Name: count, dtype: int64

Column: Copyright holder unique code
Copyright holder unique code
<class 'float'>    1556703
Name: count, dtype: int64

Column: Country
Country
<class 'str'>    1556703
Name: count, dtype: int64

Column: Currency
Currency
<class 'str'>    1556703
Name: count, dtype: int64

Column: Device
Device
<class 'float'>    1556703
Name: count, dtype: int64

Column: FX Rate
FX Rate
<class 'float'>    1556703
Name: count, dtype: int64

Column: Gross Amount
Gross Amount
<class 'float'>    1556703
Name: count, dtype: int64

Column: ISRC
ISRC
<class 'str'>    1556703
Name: count, dtype: int64

Column

In [12]:
df_final.to_csv(outputfilename, index=False)